# M1 라벨/window 검증

## tl;dr

이 노트북은 M1 `normal` vs `efd_possible` 분류에서 라벨과 window 기준이 타당한지 검증한다. 모델 고도화가 아니라, `Report date` 기준 window, `Possible anomaly start` 기준 window, Event 20/67 민감도, normal window의 disturbance-free 여부를 비교해 다음 학습 기준을 정하는 것이 목적이다.


## Context & Methods

### Key Assumptions

- M1만 사용한다. 다른 제조사 그룹은 계산, 비교, 컬럼 선정에 사용하지 않는다.
- 원본 ZIP과 metadata는 수정하지 않는다.
- 공통 센서 10개와 기존 통계 feature 생성 방식은 유지한다.
- `anomaly_original`은 길이가 서로 달라 모델 성능 비교에서 제외하고 라벨 진단용으로만 사용한다.
- 모델 비교는 라벨/window 후보 간 상대 비교용이며, 운영 모델 성능 주장이 아니다.


In [1]:
from __future__ import annotations

from pathlib import Path
import zipfile

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
DATA_DIR = ROOT / "05_데이터셋" / "PreDist"
META_DIR = DATA_DIR / "metadata" / "manufacturer_1"
ZIP_PATH = DATA_DIR / "predist_dataset.zip"
OUTPUT_DIR = ROOT / "07_데이터산출물"

AUDIT_PATH = OUTPUT_DIR / "m1_label_window_audit.csv"
SUMMARY_PATH = OUTPUT_DIR / "m1_window_candidate_summary.csv"
METRICS_PATH = OUTPUT_DIR / "m1_window_candidate_cv_metrics.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "m1_window_candidate_cv_predictions.csv"
DECISION_PATH = OUTPUT_DIR / "m1_window_decision_matrix.csv"
REPORT_PATH = OUTPUT_DIR / "04_M1_라벨_window_검증_보고서.md"

RANDOM_STATE = 42
POSITIVE_LABEL = "efd_possible"

COMMON_SENSOR_COLUMNS = [
    "outdoor_temperature",
    "s_hc1_supply_temperature",
    "s_hc1_supply_temperature_setpoint",
    "p_hc1_return_temperature",
    "p_net_meter_energy",
    "p_net_meter_volume",
    "p_net_meter_heat_power",
    "p_net_meter_flow",
    "p_net_supply_temperature",
    "p_net_return_temperature",
]
SENSOR_STATS = ["mean", "std", "min", "max", "median", "last_minus_first", "missing_rate"]


## Data

### 1. Load M1 Metadata and Strict Labels

In [2]:
faults = pd.read_csv(META_DIR / "faults.csv", sep=";")
normal_events = pd.read_csv(META_DIR / "normal_events.csv", sep=";")
disturbances = pd.read_csv(META_DIR / "disturbances.csv", sep=";")
feature_descriptions = pd.read_csv(META_DIR / "feature_descriptions.csv", sep=";")

for column in ["Report date", "Possible anomaly start", "Possible anomaly end", "Training start", "Training end"]:
    faults[column] = pd.to_datetime(faults[column], errors="coerce")
for column in ["Event start", "Event end", "Training start", "Training end"]:
    normal_events[column] = pd.to_datetime(normal_events[column], errors="coerce")
disturbances["Event start"] = pd.to_datetime(disturbances["Event start"], errors="coerce")

strict_positive = faults[
    (faults["efd_possible"].astype(str).str.strip().str.lower() == "true")
    & faults["Possible anomaly start"].notna()
    & faults["Possible anomaly end"].notna()
    & faults["Training start"].notna()
    & faults["Training end"].notna()
].copy()

assert len(normal_events) == 35
assert len(strict_positive) == 15

pd.DataFrame([
    {"item": "normal_events", "count": len(normal_events)},
    {"item": "faults", "count": len(faults)},
    {"item": "strict_positive", "count": len(strict_positive)},
])


,item,count
0,normal_events,35
1,faults,33
2,strict_positive,15


In [3]:
strict_positive = strict_positive.copy()
strict_positive["anomaly_start_to_report_days"] = (
    strict_positive["Report date"] - strict_positive["Possible anomaly start"]
).dt.total_seconds() / 86400
strict_positive["anomaly_duration_days"] = (
    strict_positive["Possible anomaly end"] - strict_positive["Possible anomaly start"]
).dt.total_seconds() / 86400
strict_positive["training_gap_to_report_days"] = (
    strict_positive["Report date"] - strict_positive["Training end"]
).dt.total_seconds() / 86400

strict_positive[[
    "Event ID", "substation ID", "Report date", "Possible anomaly start", "Possible anomaly end",
    "anomaly_start_to_report_days", "anomaly_duration_days", "training_gap_to_report_days"
]].sort_values("Event ID")


,Event ID,substation ID,Report date,Possible anomaly start,Possible anomaly end,anomaly_start_to_report_days,anomaly_duration_days,training_gap_to_report_days
0,1,10,2014-05-04 14:44:00,2014-05-03 16:00:00,2014-05-05 04:00:00,0.947222,1.500000,14.000000
1,3,12,2015-12-01 10:56:00,2015-11-29 12:00:00,2015-12-02 10:56:00,1.955556,2.955556,14.000000
4,7,26,2020-06-13 10:38:00,2020-06-12 12:00:00,2020-06-14 10:38:00,0.943056,1.943056,14.000000
5,10,30,2020-03-16 15:04:00,2020-03-12 19:00:00,2020-03-22 22:45:31,3.836111,10.156609,14.000000
8,15,29,2020-03-07 07:59:43,2020-03-06 12:00:00,2020-03-08 07:59:43,0.833137,1.833137,14.000000
9,20,11,2018-07-02 10:49:00,2018-06-29 05:00:00,2018-07-02 19:02:04,3.242361,3.584769,14.000000
18,40,24,2016-04-07 07:47:00,2016-04-05 13:00:00,2016-04-08 07:47:00,1.782639,2.782639,14.000000
19,44,8,2018-04-27 08:15:00,2018-04-26 06:00:00,2018-04-30 09:45:07,1.093750,4.156331,14.000000
21,47,28,2020-01-13 10:55:00,2020-01-08 23:00:00,2020-01-14 10:47:32,4.496528,5.491343,14.000000
22,49,18,2019-05-04 07:19:00,2019-05-03 11:00:00,2019-05-05 07:19:00,0.846528,1.846528,14.000000


### 2. Define Window Candidates

In [4]:
NORMAL_POLICIES = {
    "normal_7d": 7,
    "normal_3d": 3,
    "normal_1d": 1,
}

POSITIVE_POLICIES = {
    "report_pre_7d": {"days": 7, "model_compare": True},
    "report_pre_3d": {"days": 3, "model_compare": True},
    "report_pre_1d": {"days": 1, "model_compare": True},
    "anomaly_start_plus_7d": {"days": 7, "model_compare": True},
    "anomaly_start_pre_7d": {"days": 7, "model_compare": True},
    "anomaly_original": {"days": None, "model_compare": False},
}

SCENARIOS = {
    "all": set(),
    "no_event67": {67},
    "no_low_coverage20": {20},
    "no_event67_no_event20": {67, 20},
}

MODEL_CANDIDATES = [
    {"candidate_id": "report_pre_7d", "normal_policy": "normal_7d", "positive_policy": "report_pre_7d"},
    {"candidate_id": "report_pre_3d", "normal_policy": "normal_3d", "positive_policy": "report_pre_3d"},
    {"candidate_id": "report_pre_1d", "normal_policy": "normal_1d", "positive_policy": "report_pre_1d"},
    {"candidate_id": "anomaly_start_plus_7d", "normal_policy": "normal_7d", "positive_policy": "anomaly_start_plus_7d"},
    {"candidate_id": "anomaly_start_pre_7d", "normal_policy": "normal_7d", "positive_policy": "anomaly_start_pre_7d"},
]

pd.DataFrame(MODEL_CANDIDATES)


,candidate_id,normal_policy,positive_policy
0,report_pre_7d,normal_7d,report_pre_7d
1,report_pre_3d,normal_3d,report_pre_3d
2,report_pre_1d,normal_1d,report_pre_1d
3,anomaly_start_plus_7d,normal_7d,anomaly_start_plus_7d
4,anomaly_start_pre_7d,normal_7d,anomaly_start_pre_7d


### 3. Utility Functions

In [5]:
def normal_window(row: pd.Series, policy: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    days = NORMAL_POLICIES[policy]
    start = row["Event start"]
    return start, start + pd.Timedelta(days=days)


def positive_window(row: pd.Series, policy: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    if policy == "report_pre_7d":
        end = row["Report date"]
        return end - pd.Timedelta(days=7), end
    if policy == "report_pre_3d":
        end = row["Report date"]
        return end - pd.Timedelta(days=3), end
    if policy == "report_pre_1d":
        end = row["Report date"]
        return end - pd.Timedelta(days=1), end
    if policy == "anomaly_start_plus_7d":
        start = row["Possible anomaly start"]
        return start, start + pd.Timedelta(days=7)
    if policy == "anomaly_start_pre_7d":
        end = row["Possible anomaly start"]
        return end - pd.Timedelta(days=7), end
    if policy == "anomaly_original":
        return row["Possible anomaly start"], row["Possible anomaly end"]
    raise ValueError(policy)


def disturbance_summary(substation_id: int, start: pd.Timestamp, end: pd.Timestamp) -> tuple[int, str]:
    mask = (
        (disturbances["substation ID"].astype(int) == int(substation_id))
        & (disturbances["Event start"] >= start)
        & (disturbances["Event start"] < end)
    )
    selected = disturbances.loc[mask]
    types = ",".join(f"{key}:{value}" for key, value in selected["type"].value_counts().sort_index().items())
    return int(len(selected)), types


def load_substation_frame(zf: zipfile.ZipFile, substation_id: int) -> pd.DataFrame:
    path = f"manufacturer 1/operational_data/substation_{int(substation_id)}.csv"
    with zf.open(path) as file:
        frame = pd.read_csv(file, sep=";", usecols=["timestamp", *COMMON_SENSOR_COLUMNS])
    frame["timestamp"] = pd.to_datetime(frame["timestamp"])
    for column in COMMON_SENSOR_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame


def get_substation_frame(cache: dict[int, pd.DataFrame], zf: zipfile.ZipFile, substation_id: int) -> pd.DataFrame:
    substation_id = int(substation_id)
    if substation_id not in cache:
        cache[substation_id] = load_substation_frame(zf, substation_id)
    return cache[substation_id]


def window_slice(frame: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    return frame[(frame["timestamp"] >= start) & (frame["timestamp"] < end)]


def sensor_stats(window: pd.DataFrame) -> dict:
    output = {}
    sample_count = int(len(window))
    for column in COMMON_SENSOR_COLUMNS:
        series = window[column] if sample_count else pd.Series(dtype="float64")
        valid = series.dropna()
        output[f"{column}__mean"] = float(series.mean()) if sample_count else np.nan
        output[f"{column}__std"] = float(series.std()) if sample_count else np.nan
        output[f"{column}__min"] = float(series.min()) if sample_count else np.nan
        output[f"{column}__max"] = float(series.max()) if sample_count else np.nan
        output[f"{column}__median"] = float(series.median()) if sample_count else np.nan
        output[f"{column}__last_minus_first"] = float(valid.iloc[-1] - valid.iloc[0]) if len(valid) >= 2 else np.nan
        output[f"{column}__missing_rate"] = float(series.isna().mean()) if sample_count else np.nan
    return output


def expected_row_count(start: pd.Timestamp, end: pd.Timestamp) -> float:
    return (end - start).total_seconds() / 600


## Results

### 4. Build Label/Window Audit

In [6]:
def build_audit_rows() -> list[dict]:
    rows = []
    with zipfile.ZipFile(ZIP_PATH) as zf:
        cache: dict[int, pd.DataFrame] = {}

        for policy in NORMAL_POLICIES:
            for _, row in normal_events.iterrows():
                start, end = normal_window(row, policy)
                substation_id = int(row["substation ID"])
                frame = get_substation_frame(cache, zf, substation_id)
                selected = window_slice(frame, start, end)
                disturbance_count, disturbance_types = disturbance_summary(substation_id, start, end)
                expected_rows = expected_row_count(start, end)
                rows.append({
                    "label": "normal",
                    "policy": policy,
                    "source_event_id": int(row["Event ID"]),
                    "substation_id": substation_id,
                    "window_start": start,
                    "window_end": end,
                    "window_days": (end - start).total_seconds() / 86400,
                    "sample_count": int(len(selected)),
                    "expected_sample_count": expected_rows,
                    "coverage_rate": int(len(selected)) / expected_rows if expected_rows else np.nan,
                    "disturbance_count": disturbance_count,
                    "disturbance_types": disturbance_types,
                    "anomaly_start_to_report_days": np.nan,
                    "anomaly_duration_days": np.nan,
                    "training_gap_to_report_days": np.nan,
                })

        for policy in POSITIVE_POLICIES:
            for _, row in strict_positive.iterrows():
                start, end = positive_window(row, policy)
                substation_id = int(row["substation ID"])
                frame = get_substation_frame(cache, zf, substation_id)
                selected = window_slice(frame, start, end)
                disturbance_count, disturbance_types = disturbance_summary(substation_id, start, end)
                expected_rows = expected_row_count(start, end)
                rows.append({
                    "label": "efd_possible",
                    "policy": policy,
                    "source_event_id": int(row["Event ID"]),
                    "substation_id": substation_id,
                    "window_start": start,
                    "window_end": end,
                    "window_days": (end - start).total_seconds() / 86400,
                    "sample_count": int(len(selected)),
                    "expected_sample_count": expected_rows,
                    "coverage_rate": int(len(selected)) / expected_rows if expected_rows else np.nan,
                    "disturbance_count": disturbance_count,
                    "disturbance_types": disturbance_types,
                    "anomaly_start_to_report_days": row["anomaly_start_to_report_days"],
                    "anomaly_duration_days": row["anomaly_duration_days"],
                    "training_gap_to_report_days": row["training_gap_to_report_days"],
                })
    return rows


label_window_audit = pd.DataFrame(build_audit_rows())
for column in ["window_start", "window_end"]:
    label_window_audit[column] = pd.to_datetime(label_window_audit[column]).dt.strftime("%Y-%m-%d %H:%M:%S")
label_window_audit.to_csv(AUDIT_PATH, index=False, encoding="utf-8-sig")

label_window_audit.groupby(["label", "policy"]).agg(
    rows=("source_event_id", "count"),
    coverage_min=("coverage_rate", "min"),
    coverage_median=("coverage_rate", "median"),
    coverage_max=("coverage_rate", "max"),
    disturbance_sum=("disturbance_count", "sum"),
).round(4)


rows  coverage_min  coverage_median  \
label        policy                                                       
efd_possible anomaly_original         15        0.4630           1.0003   
             anomaly_start_plus_7d    15        0.7242           1.0000   
             anomaly_start_pre_7d     15        0.9950           1.0000   
             report_pre_1d            15        0.0972           1.0000   
             report_pre_3d            15        0.3565           1.0000   
             report_pre_7d            15        0.7242           1.0000   
normal       normal_1d                35        1.0000           1.0000   
             normal_3d                35        1.0000           1.0000   
             normal_7d                35        1.0000           1.0000   

                                    coverage_max  disturbance_sum  
label        policy                                                
efd_possible anomaly_original             1.0022               25  
             anomaly_start_plus_7d        1.0000               23  
             anomaly_start_pre_7d         1.0000                3  
             report_pre_1d                1.0000                2  
             report_pre_3d                1.0000                4  
             report_pre_7d                1.0000                5  
normal       normal_1d                    1.0000                0  
             normal_3d                    1.0000                0  
             normal_7d                    1.0000                0

In [7]:
normal_audit = label_window_audit[label_window_audit["label"] == "normal"]
positive_audit = label_window_audit[label_window_audit["label"] == "efd_possible"]

assert normal_audit["disturbance_count"].sum() == 0
assert len(label_window_audit) == (len(NORMAL_POLICIES) * 35 + len(POSITIVE_POLICIES) * 15)

positive_audit[(positive_audit["coverage_rate"] < 0.95) | (positive_audit["source_event_id"].isin([20, 67]))][[
    "label", "policy", "source_event_id", "substation_id", "window_days", "sample_count",
    "coverage_rate", "disturbance_count", "disturbance_types", "anomaly_start_to_report_days"
]].sort_values(["source_event_id", "policy"])


,label,policy,source_event_id,substation_id,window_days,sample_count,coverage_rate,disturbance_count,disturbance_types,anomaly_start_to_report_days
185,efd_possible,anomaly_original,20,11,3.584769,239,0.462993,2,"fault:1,task:1",3.242361
155,efd_possible,anomaly_start_plus_7d,20,11,7.000000,730,0.724206,2,"fault:1,task:1",3.242361
170,efd_possible,anomaly_start_pre_7d,20,11,7.000000,1008,1.000000,0,,3.242361
140,efd_possible,report_pre_1d,20,11,1.000000,14,0.097222,0,,3.242361
125,efd_possible,report_pre_3d,20,11,3.000000,154,0.356481,0,,3.242361
110,efd_possible,report_pre_7d,20,11,7.000000,730,0.724206,0,,3.242361
194,efd_possible,anomaly_original,67,7,218.510417,31466,1.000016,1,fault:1,217.510417
164,efd_possible,anomaly_start_plus_7d,67,7,7.000000,1008,1.000000,0,,217.510417
179,efd_possible,anomaly_start_pre_7d,67,7,7.000000,1008,1.000000,0,,217.510417
149,efd_possible,report_pre_1d,67,7,1.000000,144,1.000000,0,,217.510417


### 5. Build Candidate Features and Run Group CV

In [8]:
def build_candidate_features(candidate: dict, scenario_name: str, excluded_event_ids: set[int]) -> pd.DataFrame:
    rows = []
    with zipfile.ZipFile(ZIP_PATH) as zf:
        cache: dict[int, pd.DataFrame] = {}

        for _, row in normal_events.iterrows():
            start, end = normal_window(row, candidate["normal_policy"])
            substation_id = int(row["substation ID"])
            frame = get_substation_frame(cache, zf, substation_id)
            selected = window_slice(frame, start, end)
            disturbance_count, disturbance_types = disturbance_summary(substation_id, start, end)
            expected_rows = expected_row_count(start, end)
            output = {
                "candidate_id": candidate["candidate_id"],
                "scenario": scenario_name,
                "normal_policy": candidate["normal_policy"],
                "positive_policy": candidate["positive_policy"],
                "sample_id": f"{candidate['candidate_id']}__{scenario_name}__normal_{int(row['Event ID']):04d}",
                "label": "normal",
                "source_event_id": int(row["Event ID"]),
                "substation_id": substation_id,
                "window_start": start,
                "window_end": end,
                "window_days": (end - start).total_seconds() / 86400,
                "sample_count": int(len(selected)),
                "expected_sample_count": expected_rows,
                "coverage_rate": int(len(selected)) / expected_rows if expected_rows else np.nan,
                "disturbance_count": disturbance_count,
                "disturbance_types": disturbance_types,
            }
            output.update(sensor_stats(selected))
            rows.append(output)

        for _, row in strict_positive.iterrows():
            event_id = int(row["Event ID"])
            if event_id in excluded_event_ids:
                continue
            start, end = positive_window(row, candidate["positive_policy"])
            substation_id = int(row["substation ID"])
            frame = get_substation_frame(cache, zf, substation_id)
            selected = window_slice(frame, start, end)
            disturbance_count, disturbance_types = disturbance_summary(substation_id, start, end)
            expected_rows = expected_row_count(start, end)
            output = {
                "candidate_id": candidate["candidate_id"],
                "scenario": scenario_name,
                "normal_policy": candidate["normal_policy"],
                "positive_policy": candidate["positive_policy"],
                "sample_id": f"{candidate['candidate_id']}__{scenario_name}__efd_possible_{event_id:04d}",
                "label": "efd_possible",
                "source_event_id": event_id,
                "substation_id": substation_id,
                "window_start": start,
                "window_end": end,
                "window_days": (end - start).total_seconds() / 86400,
                "sample_count": int(len(selected)),
                "expected_sample_count": expected_rows,
                "coverage_rate": int(len(selected)) / expected_rows if expected_rows else np.nan,
                "disturbance_count": disturbance_count,
                "disturbance_types": disturbance_types,
                "anomaly_start_to_report_days": row["anomaly_start_to_report_days"],
                "anomaly_duration_days": row["anomaly_duration_days"],
                "training_gap_to_report_days": row["training_gap_to_report_days"],
            }
            output.update(sensor_stats(selected))
            rows.append(output)

    features = pd.DataFrame(rows)
    for column in ["window_start", "window_end"]:
        features[column] = pd.to_datetime(features[column]).dt.strftime("%Y-%m-%d %H:%M:%S")
    return features


def score_fold(model_name: str, y_true: pd.Series, y_pred: np.ndarray, y_score: np.ndarray | None) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    output = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": np.nan,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }
    if y_score is not None and y_true.nunique() == 2:
        output["roc_auc"] = roc_auc_score(y_true, y_score)
    return output


def evaluate_candidate(features: pd.DataFrame) -> tuple[list[dict], pd.DataFrame]:
    feature_columns = [column for column in features.columns if "__" in column]
    assert len(feature_columns) == len(COMMON_SENSOR_COLUMNS) * len(SENSOR_STATS)
    assert {"substation_id", "source_event_id", "coverage_rate"}.isdisjoint(feature_columns)

    X = features[feature_columns]
    y = (features["label"] == POSITIVE_LABEL).astype(int)
    groups = features["substation_id"].astype(int)
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(X, y, groups))

    models = {
        "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
        "logistic_balanced": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
        ]),
    }

    metric_rows = []
    prediction_rows = []
    for fold, (train_index, test_index) in enumerate(splits, start=1):
        train_groups = set(groups.iloc[train_index])
        test_groups = set(groups.iloc[test_index])
        assert train_groups.isdisjoint(test_groups)

        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]
        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]

        for model_name, model in models.items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            y_score = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
            scores = score_fold(model_name, y_test, y_pred, y_score)
            scores.update({
                "candidate_id": features["candidate_id"].iloc[0],
                "scenario": features["scenario"].iloc[0],
                "normal_policy": features["normal_policy"].iloc[0],
                "positive_policy": features["positive_policy"].iloc[0],
                "fold": fold,
                "train_rows": int(len(train_index)),
                "test_rows": int(len(test_index)),
                "train_groups": int(len(train_groups)),
                "test_groups": int(len(test_groups)),
                "test_normal": int((y_test == 0).sum()),
                "test_efd_possible": int((y_test == 1).sum()),
            })
            metric_rows.append(scores)

            predictions = features.iloc[test_index][[
                "candidate_id", "scenario", "normal_policy", "positive_policy", "sample_id", "label",
                "substation_id", "source_event_id", "window_start", "window_end", "window_days",
                "coverage_rate", "disturbance_count", "disturbance_types",
            ]].copy()
            predictions["fold"] = fold
            predictions["model"] = model_name
            predictions["actual_binary"] = y_test.to_numpy()
            predictions["predicted_binary"] = y_pred
            predictions["predicted_label"] = np.where(y_pred == 1, "efd_possible", "normal")
            predictions["positive_score"] = y_score if y_score is not None else np.nan
            predictions["is_correct"] = predictions["actual_binary"] == predictions["predicted_binary"]
            prediction_rows.append(predictions)

    return metric_rows, pd.concat(prediction_rows, ignore_index=True)


In [9]:
candidate_summary_rows = []
all_metric_rows = []
all_prediction_frames = []

for candidate in MODEL_CANDIDATES:
    for scenario_name, excluded_event_ids in SCENARIOS.items():
        candidate_features = build_candidate_features(candidate, scenario_name, excluded_event_ids)
        metric_rows, predictions = evaluate_candidate(candidate_features)
        all_metric_rows.extend(metric_rows)
        all_prediction_frames.append(predictions)

        label_counts = candidate_features["label"].value_counts().to_dict()
        positive_events = sorted(candidate_features.loc[candidate_features["label"] == "efd_possible", "source_event_id"].astype(int).tolist())
        candidate_summary_rows.append({
            "candidate_id": candidate["candidate_id"],
            "scenario": scenario_name,
            "normal_policy": candidate["normal_policy"],
            "positive_policy": candidate["positive_policy"],
            "rows": int(len(candidate_features)),
            "normal_rows": int(label_counts.get("normal", 0)),
            "positive_rows": int(label_counts.get("efd_possible", 0)),
            "positive_event_ids": ",".join(str(value) for value in positive_events),
            "window_days_min": candidate_features["window_days"].min(),
            "window_days_median": candidate_features["window_days"].median(),
            "window_days_max": candidate_features["window_days"].max(),
            "coverage_min": candidate_features["coverage_rate"].min(),
            "coverage_median": candidate_features["coverage_rate"].median(),
            "coverage_max": candidate_features["coverage_rate"].max(),
            "low_coverage_count_lt_095": int((candidate_features["coverage_rate"] < 0.95).sum()),
            "normal_disturbance_sum": int(candidate_features.loc[candidate_features["label"] == "normal", "disturbance_count"].sum()),
            "positive_disturbance_sum": int(candidate_features.loc[candidate_features["label"] == "efd_possible", "disturbance_count"].sum()),
        })

candidate_summary = pd.DataFrame(candidate_summary_rows)
cv_metrics = pd.DataFrame(all_metric_rows)
cv_predictions = pd.concat(all_prediction_frames, ignore_index=True)

candidate_summary.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
cv_metrics.to_csv(METRICS_PATH, index=False, encoding="utf-8-sig")
cv_predictions.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

candidate_summary.head()


,candidate_id,scenario,normal_policy,positive_policy,rows,normal_rows,positive_rows,positive_event_ids,window_days_min,window_days_median,window_days_max,coverage_min,coverage_median,coverage_max,low_coverage_count_lt_095,normal_disturbance_sum,positive_disturbance_sum
0,report_pre_7d,all,normal_7d,report_pre_7d,50,35,15,"1,3,7,10,15,20,40,44,47,49,52,53,57,64,67",7.0,7.0,7.0,0.724206,1.0,1.0,1,0,5
1,report_pre_7d,no_event67,normal_7d,report_pre_7d,49,35,14,"1,3,7,10,15,20,40,44,47,49,52,53,57,64",7.0,7.0,7.0,0.724206,1.0,1.0,1,0,5
2,report_pre_7d,no_low_coverage20,normal_7d,report_pre_7d,49,35,14,"1,3,7,10,15,40,44,47,49,52,53,57,64,67",7.0,7.0,7.0,0.995040,1.0,1.0,0,0,5
3,report_pre_7d,no_event67_no_event20,normal_7d,report_pre_7d,48,35,13,"1,3,7,10,15,40,44,47,49,52,53,57,64",7.0,7.0,7.0,0.995040,1.0,1.0,0,0,5
4,report_pre_3d,all,normal_3d,report_pre_3d,50,35,15,"1,3,7,10,15,20,40,44,47,49,52,53,57,64,67",3.0,3.0,3.0,0.356481,1.0,1.0,1,0,4


### 6. Compare Candidate Windows

In [10]:
metric_summary = cv_metrics.groupby(["candidate_id", "scenario", "model"]).agg(
    accuracy_mean=("accuracy", "mean"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    precision_mean=("precision", "mean"),
    recall_mean=("recall", "mean"),
    f1_mean=("f1", "mean"),
    roc_auc_mean=("roc_auc", "mean"),
    fp_sum=("fp", "sum"),
    fn_sum=("fn", "sum"),
).reset_index()

logistic_summary = metric_summary[metric_summary["model"] == "logistic_balanced"].copy()
dummy_summary = metric_summary[metric_summary["model"] == "dummy_most_frequent"][[
    "candidate_id", "scenario", "balanced_accuracy_mean", "recall_mean", "f1_mean"
]].rename(columns={
    "balanced_accuracy_mean": "dummy_balanced_accuracy_mean",
    "recall_mean": "dummy_recall_mean",
    "f1_mean": "dummy_f1_mean",
})

decision_matrix = candidate_summary.merge(logistic_summary, on=["candidate_id", "scenario"], how="left")
decision_matrix = decision_matrix.merge(dummy_summary, on=["candidate_id", "scenario"], how="left")
decision_matrix["balanced_accuracy_lift_vs_dummy"] = (
    decision_matrix["balanced_accuracy_mean"] - decision_matrix["dummy_balanced_accuracy_mean"]
)
decision_matrix["recall_lift_vs_dummy"] = decision_matrix["recall_mean"] - decision_matrix["dummy_recall_mean"]
decision_matrix["coverage_status"] = np.where(decision_matrix["coverage_min"] >= 0.95, "ok", "low_coverage_present")
decision_matrix["normal_label_status"] = np.where(
    decision_matrix["normal_disturbance_sum"] == 0, "normal_clean", "normal_disturbance_overlap"
)
decision_matrix["event20_included"] = decision_matrix["positive_event_ids"].str.contains("20")
decision_matrix["event67_included"] = decision_matrix["positive_event_ids"].str.contains("67")
decision_matrix["outlier_status"] = np.select(
    [
        decision_matrix["event20_included"] & decision_matrix["event67_included"],
        decision_matrix["event20_included"] & ~decision_matrix["event67_included"],
        ~decision_matrix["event20_included"] & decision_matrix["event67_included"],
    ],
    ["event20_and_event67_included", "event20_included", "event67_included"],
    default="event20_event67_excluded",
)
decision_matrix["fp_rate_over_normal"] = decision_matrix["fp_sum"] / decision_matrix["normal_rows"]
decision_matrix["fn_rate_over_positive"] = decision_matrix["fn_sum"] / decision_matrix["positive_rows"]


def hold_reason(row: pd.Series) -> str:
    reasons = []
    if row["coverage_status"] != "ok":
        reasons.append("coverage<0.95")
    if row["positive_disturbance_sum"] > 0 and "plus" in row["candidate_id"]:
        reasons.append("post/anomaly window disturbance risk")
    if row["fp_rate_over_normal"] > 0.35:
        reasons.append("high false positives")
    if row["balanced_accuracy_lift_vs_dummy"] <= 0:
        reasons.append("no lift vs dummy")
    return "; ".join(reasons) if reasons else "reviewable"


decision_matrix["review_status"] = decision_matrix.apply(hold_reason, axis=1)
decision_matrix.to_csv(DECISION_PATH, index=False, encoding="utf-8-sig")

decision_matrix.sort_values(["scenario", "balanced_accuracy_mean"], ascending=[True, False]).head(12)


,candidate_id,scenario,normal_policy,positive_policy,rows,normal_rows,positive_rows,positive_event_ids,window_days_min,window_days_median,...,balanced_accuracy_lift_vs_dummy,recall_lift_vs_dummy,coverage_status,normal_label_status,event20_included,event67_included,outlier_status,fp_rate_over_normal,fn_rate_over_positive,review_status
12,anomaly_start_plus_7d,all,normal_7d,anomaly_start_plus_7d,50,35,15,"1,3,7,10,15,20,40,44,47,49,52,53,57,64,67",7.0,7.0,...,0.138095,0.600000,low_coverage_present,normal_clean,True,True,event20_and_event67_included,0.314286,0.400000,coverage<0.95; post/anomaly window disturbance...
0,report_pre_7d,all,normal_7d,report_pre_7d,50,35,15,"1,3,7,10,15,20,40,44,47,49,52,53,57,64,67",7.0,7.0,...,0.094643,0.600000,low_coverage_present,normal_clean,True,True,event20_and_event67_included,0.400000,0.400000,coverage<0.95; high false positives
4,report_pre_3d,all,normal_3d,report_pre_3d,50,35,15,"1,3,7,10,15,20,40,44,47,49,52,53,57,64,67",3.0,3.0,...,0.015476,0.333333,low_coverage_present,normal_clean,True,True,event20_and_event67_included,0.285714,0.666667,coverage<0.95
8,report_pre_1d,all,normal_1d,report_pre_1d,50,35,15,"1,3,7,10,15,20,40,44,47,49,52,53,57,64,67",1.0,1.0,...,-0.028571,0.266667,low_coverage_present,normal_clean,True,True,event20_and_event67_included,0.314286,0.733333,coverage<0.95; no lift vs dummy
16,anomaly_start_pre_7d,all,normal_7d,anomaly_start_pre_7d,50,35,15,"1,3,7,10,15,20,40,44,47,49,52,53,57,64,67",7.0,7.0,...,-0.030357,0.466667,ok,normal_clean,True,True,event20_and_event67_included,0.514286,0.533333,high false positives; no lift vs dummy
13,anomaly_start_plus_7d,no_event67,normal_7d,anomaly_start_plus_7d,49,35,14,"1,3,7,10,15,20,40,44,47,49,52,53,57,64",7.0,7.0,...,0.173214,0.666667,low_coverage_present,normal_clean,True,False,event20_included,0.314286,0.357143,coverage<0.95; post/anomaly window disturbance...
1,report_pre_7d,no_event67,normal_7d,report_pre_7d,49,35,14,"1,3,7,10,15,20,40,44,47,49,52,53,57,64",7.0,7.0,...,0.167262,0.666667,low_coverage_present,normal_clean,True,False,event20_included,0.342857,0.357143,coverage<0.95
9,report_pre_1d,no_event67,normal_1d,report_pre_1d,49,35,14,"1,3,7,10,15,20,40,44,47,49,52,53,57,64",1.0,1.0,...,0.041667,0.300000,low_coverage_present,normal_clean,True,False,event20_included,0.228571,0.714286,coverage<0.95
5,report_pre_3d,no_event67,normal_3d,report_pre_3d,49,35,14,"1,3,7,10,15,20,40,44,47,49,52,53,57,64",3.0,3.0,...,-0.004167,0.233333,low_coverage_present,normal_clean,True,False,event20_included,0.257143,0.785714,coverage<0.95; no lift vs dummy
17,anomaly_start_pre_7d,no_event67,normal_7d,anomaly_start_pre_7d,49,35,14,"1,3,7,10,15,20,40,44,47,49,52,53,57,64",7.0,7.0,...,-0.042857,0.300000,ok,normal_clean,True,False,event20_included,0.371429,0.714286,high false positives; no lift vs dummy


In [11]:
best_by_scenario = decision_matrix.sort_values(
    ["scenario", "balanced_accuracy_mean", "f1_mean", "recall_mean"],
    ascending=[True, False, False, False],
).groupby("scenario").head(1).reset_index(drop=True)

reviewable = decision_matrix[
    (decision_matrix["coverage_status"] == "ok")
    & (decision_matrix["balanced_accuracy_lift_vs_dummy"] > 0)
].copy()
reviewable = reviewable.sort_values(
    ["balanced_accuracy_mean", "f1_mean", "fp_rate_over_normal"],
    ascending=[False, False, True],
)

best_by_scenario


,candidate_id,scenario,normal_policy,positive_policy,rows,normal_rows,positive_rows,positive_event_ids,window_days_min,window_days_median,...,balanced_accuracy_lift_vs_dummy,recall_lift_vs_dummy,coverage_status,normal_label_status,event20_included,event67_included,outlier_status,fp_rate_over_normal,fn_rate_over_positive,review_status
0,anomaly_start_plus_7d,all,normal_7d,anomaly_start_plus_7d,50,35,15,"1,3,7,10,15,20,40,44,47,49,52,53,57,64,67",7.0,7.0,...,0.138095,0.600000,low_coverage_present,normal_clean,True,True,event20_and_event67_included,0.314286,0.400000,coverage<0.95; post/anomaly window disturbance...
1,anomaly_start_plus_7d,no_event67,normal_7d,anomaly_start_plus_7d,49,35,14,"1,3,7,10,15,20,40,44,47,49,52,53,57,64",7.0,7.0,...,0.173214,0.666667,low_coverage_present,normal_clean,True,False,event20_included,0.314286,0.357143,coverage<0.95; post/anomaly window disturbance...
2,report_pre_7d,no_event67_no_event20,normal_7d,report_pre_7d,48,35,13,"1,3,7,10,15,40,44,47,49,52,53,57,64",7.0,7.0,...,0.098810,0.366667,ok,normal_clean,False,False,event20_event67_excluded,0.171429,0.615385,reviewable
3,report_pre_7d,no_low_coverage20,normal_7d,report_pre_7d,49,35,14,"1,3,7,10,15,40,44,47,49,52,53,57,64,67",7.0,7.0,...,0.104762,0.500000,ok,normal_clean,False,True,event67_included,0.285714,0.500000,reviewable


In [12]:
reviewable.head(10)

,candidate_id,scenario,normal_policy,positive_policy,rows,normal_rows,positive_rows,positive_event_ids,window_days_min,window_days_median,...,balanced_accuracy_lift_vs_dummy,recall_lift_vs_dummy,coverage_status,normal_label_status,event20_included,event67_included,outlier_status,fp_rate_over_normal,fn_rate_over_positive,review_status
2,report_pre_7d,no_low_coverage20,normal_7d,report_pre_7d,49,35,14,"1,3,7,10,15,40,44,47,49,52,53,57,64,67",7.0,7.0,...,0.104762,0.500000,ok,normal_clean,False,True,event67_included,0.285714,0.500000,reviewable
3,report_pre_7d,no_event67_no_event20,normal_7d,report_pre_7d,48,35,13,"1,3,7,10,15,40,44,47,49,52,53,57,64",7.0,7.0,...,0.098810,0.366667,ok,normal_clean,False,False,event20_event67_excluded,0.171429,0.615385,reviewable
10,report_pre_1d,no_low_coverage20,normal_1d,report_pre_1d,49,35,14,"1,3,7,10,15,40,44,47,49,52,53,57,64,67",1.0,1.0,...,0.077381,0.433333,ok,normal_clean,False,True,event67_included,0.285714,0.571429,reviewable
11,report_pre_1d,no_event67_no_event20,normal_1d,report_pre_1d,48,35,13,"1,3,7,10,15,40,44,47,49,52,53,57,64",1.0,1.0,...,0.051786,0.333333,ok,normal_clean,False,False,event20_event67_excluded,0.228571,0.692308,reviewable
7,report_pre_3d,no_event67_no_event20,normal_3d,report_pre_3d,48,35,13,"1,3,7,10,15,40,44,47,49,52,53,57,64",3.0,3.0,...,0.016071,0.233333,ok,normal_clean,False,False,event20_event67_excluded,0.200000,0.769231,reviewable
15,anomaly_start_plus_7d,no_event67_no_event20,normal_7d,anomaly_start_plus_7d,48,35,13,"1,3,7,10,15,40,44,47,49,52,53,57,64",7.0,7.0,...,0.004167,0.333333,ok,normal_clean,False,False,event20_event67_excluded,0.314286,0.692308,post/anomaly window disturbance risk
14,anomaly_start_plus_7d,no_low_coverage20,normal_7d,anomaly_start_plus_7d,49,35,14,"1,3,7,10,15,40,44,47,49,52,53,57,64,67",7.0,7.0,...,0.002381,0.366667,ok,normal_clean,False,True,event67_included,0.342857,0.642857,post/anomaly window disturbance risk


### 7. Write Report

In [13]:
def fmt_float(value, digits=4):
    if pd.isna(value):
        return ""
    return f"{float(value):.{digits}f}"


def markdown_table(frame: pd.DataFrame) -> str:
    if frame.empty:
        return "없음"
    clean = frame.copy().where(pd.notna(frame), "")
    headers = [str(column) for column in clean.columns]
    rows = [[str(value) for value in row] for row in clean.to_numpy()]

    def escape(value: str) -> str:
        return value.replace("|", "\\|").replace("\n", "<br>")

    lines = [
        "| " + " | ".join(escape(header) for header in headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(escape(value) for value in row) + " |")
    return "\n".join(lines)


def compact_metrics(frame: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "candidate_id", "scenario", "positive_rows", "coverage_min", "positive_disturbance_sum",
        "balanced_accuracy_mean", "recall_mean", "f1_mean", "fp_sum", "fn_sum", "review_status",
    ]
    output = frame[columns].copy()
    for column in ["coverage_min", "balanced_accuracy_mean", "recall_mean", "f1_mean"]:
        output[column] = output[column].map(fmt_float)
    return output


normal_clean = int(normal_audit["disturbance_count"].sum()) == 0
event67_days = strict_positive.loc[strict_positive["Event ID"] == 67, "anomaly_start_to_report_days"].iloc[0]
event20_coverage = label_window_audit[
    (label_window_audit["label"] == "efd_possible")
    & (label_window_audit["policy"] == "report_pre_7d")
    & (label_window_audit["source_event_id"] == 20)
]["coverage_rate"].iloc[0]

best_table = compact_metrics(best_by_scenario)
reviewable_table = compact_metrics(reviewable.head(8))
report_pre7_rows = compact_metrics(decision_matrix[decision_matrix["candidate_id"] == "report_pre_7d"].sort_values("scenario"))

audit_summary = label_window_audit.groupby(["label", "policy"]).agg(
    rows=("source_event_id", "count"),
    coverage_min=("coverage_rate", "min"),
    coverage_median=("coverage_rate", "median"),
    coverage_max=("coverage_rate", "max"),
    disturbance_sum=("disturbance_count", "sum"),
).reset_index()
for column in ["coverage_min", "coverage_median", "coverage_max"]:
    audit_summary[column] = audit_summary[column].map(fmt_float)

conclusion = "현재 50행만으로는 최종 window를 확정하지 않고, Event 20/67 민감도를 반영한 보수 기준과 샘플 확장을 먼저 진행한다."
if not reviewable.empty:
    top = reviewable.iloc[0]
    conclusion += f" 가장 안정적인 후보는 `{top['candidate_id']}` / `{top['scenario']}`로 관찰됐지만, 이 결론은 탐색 결과로만 둔다."

report = f"""# M1 라벨/window 검증 보고서

## 개요

M1 `normal` vs `efd_possible` 최소학습 이후, 라벨 품질과 window 기준을 검증했다. 목표는 모델을 더 복잡하게 만드는 것이 아니라, 현재 라벨/window가 학습 기준으로 타당한지 판단하는 것이다.

## 핵심 결론

{conclusion}

- normal 35개 이벤트의 7일 구간 안에는 M1 disturbance가 없었다: `{normal_clean}`
- Event ID 20의 `report_pre_7d` coverage는 `{event20_coverage:.4f}`로 낮다.
- Event ID 67은 anomaly start가 report date보다 `{event67_days:.1f}`일 앞선 장기 anomaly다.
- 성능 숫자 하나만으로 window를 고르지 않고, coverage, disturbance overlap, outlier 민감도, group CV 결과를 함께 봐야 한다.

## 라벨/window audit 요약

{markdown_table(audit_summary)}

## 시나리오별 최고 후보

{markdown_table(best_table)}

## 검토 가능한 상위 후보

{markdown_table(reviewable_table)}

## 기존 report_pre_7d 민감도

{markdown_table(report_pre7_rows)}

## 판단

- `report_pre_7d`는 기존 기준과 맞지만 Event 20 low coverage 영향을 받는다.
- `anomaly_start_plus_7d`와 `anomaly_original` 계열은 disturbance overlap과 사후 정보 혼입 위험이 있어 모델 성능 비교 기준으로 신중해야 한다.
- Event 67은 이상 시작이 너무 오래 이어진 케이스라 일반적인 조기탐지 window 판단을 왜곡할 수 있다.
- 따라서 다음 학습 기준은 한 번에 확정하지 말고, `no_event67_no_event20` 같은 보수 버전을 함께 유지해 비교하는 것이 안전하다.

## 생성 파일

- `07_데이터산출물/m1_label_window_audit.csv`
- `07_데이터산출물/m1_window_candidate_summary.csv`
- `07_데이터산출물/m1_window_candidate_cv_metrics.csv`
- `07_데이터산출물/m1_window_candidate_cv_predictions.csv`
- `07_데이터산출물/m1_window_decision_matrix.csv`

## 한계와 주의점

- 전체 샘플이 50행 수준이라 window 결정의 통계적 안정성이 낮다.
- group CV는 누수를 줄이지만 fold별 positive 수가 적다.
- 이 결과는 다음 학습 기준을 좁히기 위한 진단 결과이며, 운영 모델 성능 근거가 아니다.
"""
REPORT_PATH.write_text(report, encoding="utf-8")

assert AUDIT_PATH.exists()
assert SUMMARY_PATH.exists()
assert METRICS_PATH.exists()
assert PREDICTIONS_PATH.exists()
assert DECISION_PATH.exists()
assert REPORT_PATH.exists()
assert normal_clean
assert len(cv_metrics) == len(MODEL_CANDIDATES) * len(SCENARIOS) * 5 * 2
assert len(cv_predictions) > 0
assert set(cv_predictions["model"]) == {"dummy_most_frequent", "logistic_balanced"}

print("audit:", AUDIT_PATH.relative_to(ROOT))
print("summary:", SUMMARY_PATH.relative_to(ROOT))
print("metrics:", METRICS_PATH.relative_to(ROOT))
print("predictions:", PREDICTIONS_PATH.relative_to(ROOT))
print("decision:", DECISION_PATH.relative_to(ROOT))
print("report:", REPORT_PATH.relative_to(ROOT))
print("validation complete")


audit: 07_데이터산출물\m1_label_window_audit.csv
summary: 07_데이터산출물\m1_window_candidate_summary.csv
metrics: 07_데이터산출물\m1_window_candidate_cv_metrics.csv
predictions: 07_데이터산출물\m1_window_candidate_cv_predictions.csv
decision: 07_데이터산출물\m1_window_decision_matrix.csv
report: 07_데이터산출물\04_M1_라벨_window_검증_보고서.md
validation complete
